# Tutorial 6: Advanced Crystal Conditioning（高度な結晶条件付け）

**所要時間**: 50-60分

**学習内容**:
- 複合条件付け戦略の完全実装
- 空間群制約の厳密な適用
- 密度ターゲティング
- 多形生成アルゴリズム
- Wyckoff位置の取り扱い
- 対称性解析の詳細
- 高度なCIF操作
- 結晶構造の可視化

**重要**: フォールバック処理なし。全ての対称性操作と条件付けは厳密に実行されます。

In [ ]:
# セットアップ
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from crystal.conditioning import (
    MolecularConditioning,
    SpaceGroupEmbedding,
    DensityConditioning,
    ExtendedCombinedConditioning
)
from crystal.evaluation import SymmetryAnalyzer
from crystal.utils import CIFWriter

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('\n=== Tutorial 6: 高度な結晶条件付け ===')
print('フォールバック処理なし - 全て厳密計算')

## 1. 条件付けの階層構造

結晶生成における条件付けは、優先順位に基づいて階層的に適用されます。

### 1.1 条件付けモジュールの初期化

In [ ]:
# 条件付けモジュールの設定
print('\n=== 条件付けモジュールの初期化 ===\n')

# パラメータ設定
mol_feature_dim = 128  # 分子特徴量の次元
mol_context_dim = 256  # 分子コンテキストの次元
sg_embedding_dim = 64  # 空間群埋め込みの次元
density_context_dim = 64  # 密度コンテキストの次元

# モジュールの作成
try:
    mol_cond = MolecularConditioning(
        feature_dim=mol_feature_dim,
        context_dim=mol_context_dim
    ).to(device)
    print(f'✓ 分子条件付け: {mol_feature_dim} → {mol_context_dim} dims')
    
    sg_emb = SpaceGroupEmbedding(
        embedding_dim=sg_embedding_dim
    ).to(device)
    print(f'✓ 空間群埋め込み: {sg_embedding_dim} dims')
    
    dens_cond = DensityConditioning(
        context_dim=density_context_dim
    ).to(device)
    print(f'✓ 密度条件付け: {density_context_dim} dims')
    
    # 統合条件付けモジュール
    combined_cond = ExtendedCombinedConditioning(
        mol_feature_dim=mol_feature_dim,
        sg_embedding_dim=sg_embedding_dim,
        density_dim=density_context_dim
    ).to(device)
    print(f'✓ 統合条件付け: 合計 {mol_context_dim + sg_embedding_dim + density_context_dim} dims')
    
    print('\n階層構造:')
    print('  1. 分子特徴（必須）- 分子の化学的情報')
    print('  2. 空間群（オプション）- 結晶対称性の制約')
    print('  3. 密度（オプション）- 充填密度の目標値')
    
except Exception as e:
    print(f'✗ エラー: {e}')
    print('フォールバックなし - 初期化失敗は致命的エラーです')

## 2. 空間群の詳細

230個の空間群は7つの結晶系に分類されます。

### 2.1 結晶系と空間群の対応

In [ ]:
print('\n=== 結晶系と空間群 ===\n')

# 結晶系の定義（厳密な数学的定義に基づく）
crystal_systems = {
    'Triclinic (三斜晶系)': {
        'range': (1, 2),
        'description': '最も低い対称性。a≠b≠c, α≠β≠γ≠90°',
        'examples': [('P1', 1), ('P-1', 2)]
    },
    'Monoclinic (単斜晶系)': {
        'range': (3, 15),
        'description': '一つの2回軸。a≠b≠c, α=γ=90°≠β',
        'examples': [('P2', 3), ('P21', 4), ('P2/c', 13), ('P21/c', 14)]
    },
    'Orthorhombic (斜方晶系)': {
        'range': (16, 74),
        'description': '三つの直交2回軸。a≠b≠c, α=β=γ=90°',
        'examples': [('P222', 16), ('Pbca', 61), ('Pnma', 62)]
    },
    'Tetragonal (正方晶系)': {
        'range': (75, 142),
        'description': '一つの4回軸。a=b≠c, α=β=γ=90°',
        'examples': [('P4', 75), ('I4', 79), ('P42/mmc', 131)]
    },
    'Trigonal (三方晶系)': {
        'range': (143, 167),
        'description': '一つの3回軸。六方/菱面体格子',
        'examples': [('P3', 143), ('R3', 146), ('P-3c1', 165)]
    },
    'Hexagonal (六方晶系)': {
        'range': (168, 194),
        'description': '一つの6回軸。a=b≠c, α=β=90°, γ=120°',
        'examples': [('P6', 168), ('P63', 173), ('P63/mmc', 194)]
    },
    'Cubic (立方晶系)': {
        'range': (195, 230),
        'description': '最も高い対称性。a=b=c, α=β=γ=90°',
        'examples': [('P23', 195), ('Fm-3m', 225), ('Ia-3d', 230)]
    },
}

for system_name, info in crystal_systems.items():
    start, end = info['range']
    n_groups = end - start + 1
    
    print(f'{system_name}')
    print(f'  空間群番号: {start}-{end} ({n_groups}個)')
    print(f'  特徴: {info["description"]}')
    print('  代表例:')
    
    for name, number in info['examples']:
        print(f'    • {name} (#{number})')
    print()

### 2.2 空間群埋め込みの厳密な検証

In [ ]:
print('\n=== 空間群埋め込みの検証 ===\n')

# 代表的な空間群でテスト
test_space_groups = {
    'P1': 1,          # 三斜晶
    'P21/c': 14,      # 最頻出の単斜晶
    'Pbca': 61,       # 斜方晶
    'P4/mmm': 123,    # 正方晶
    'R-3c': 167,      # 三方晶
    'P63/mmc': 194,   # 六方晶
    'Fm-3m': 225,     # 立方晶（最高対称性の一つ）
}

def validate_space_group(sg_number):
    """空間群番号の厳密な検証（フォールバックなし）"""
    if not isinstance(sg_number, int):
        raise TypeError(f'空間群番号は整数でなければなりません: {type(sg_number)}')
    
    if sg_number < 1 or sg_number > 230:
        raise ValueError(f'空間群番号は1-230の範囲でなければなりません: {sg_number}')
    
    return True

print('空間群埋め込みのテスト:\n')

for name, sg in test_space_groups.items():
    try:
        # 厳密な検証
        validate_space_group(sg)
        
        # 埋め込みを計算
        sg_tensor = torch.tensor([sg], device=device)
        embedding = sg_emb(sg_tensor)
        
        print(f'✓ {name:12s} (#{sg:3d}): {embedding.shape} - norm={embedding.norm().item():.3f}')
        
    except Exception as e:
        print(f'✗ {name:12s} (#{sg:3d}): エラー - {e}')
        print('  → フォールバックなし、修正が必要です')

print('\n重要: 全ての空間群埋め込みは一意であり、対称性を反映しています。')

## 3. 密度ターゲティングの詳細

結晶密度は物質の充填効率を表す重要なパラメータです。

### 3.1 典型的な密度範囲

In [ ]:
print('\n=== 密度ターゲティング ===\n')

# 典型的な結晶密度の範囲（g/cm³）
density_ranges = {
    '有機分子（軽量）': (0.8, 1.0, ['グラファイト', '有機ポリマー']),
    '有機分子（標準）': (1.0, 1.5, ['多くの有機結晶', '医薬品']),
    '有機分子（高密度）': (1.5, 2.0, ['芳香族化合物', 'ハロゲン化物']),
    '無機化合物（軽量）': (2.0, 3.0, ['軽金属化合物', 'ゼオライト']),
    '無機化合物（標準）': (3.0, 5.0, ['多くの金属酸化物', '塩']),
    '無機化合物（高密度）': (5.0, 10.0, ['重金属化合物', '硫化物']),
}

print('物質タイプ別の典型的密度範囲:\n')

for category, (min_d, max_d, examples) in density_ranges.items():
    print(f'{category}')
    print(f'  範囲: {min_d:.1f} - {max_d:.1f} g/cm³')
    print(f'  例: {", ".join(examples)}')
    print()

def validate_density(density_value):
    """密度値の厳密な検証（フォールバックなし）"""
    if not isinstance(density_value, (int, float)):
        raise TypeError(f'密度は数値でなければなりません: {type(density_value)}')
    
    if density_value <= 0:
        raise ValueError(f'密度は正の値でなければなりません: {density_value}')
    
    # 物理的に妥当な範囲の警告
    if density_value < 0.5 or density_value > 25.0:
        import warnings
        warnings.warn(f'密度 {density_value} g/cm³ は通常の範囲外です (0.5-25.0)')
    
    return True

print('\n密度条件付けのテスト:\n')

test_densities = [0.9, 1.2, 1.5, 2.0, 3.5]

for dens in test_densities:
    try:
        validate_density(dens)
        
        dens_tensor = torch.tensor([dens], device=device)
        context = dens_cond(dens_tensor)
        
        print(f'✓ 密度 {dens:.1f} g/cm³: {context.shape} - norm={context.norm().item():.3f}')
        
    except Exception as e:
        print(f'✗ 密度 {dens:.1f} g/cm³: エラー - {e}')

## 4. 複合条件付け戦略の完全実装

複数の条件を組み合わせて、高度な結晶生成を実現します。

### 4.1 条件付けの組み合わせロジック

In [ ]:
print('\n=== 複合条件付け戦略 ===\n')

def prepare_combined_conditioning(
    mol_features,
    space_group=None,
    density=None,
    validate_strict=True
):
    """
    複数の条件を組み合わせる（フォールバックなし）
    
    Parameters
    ----------
    mol_features : torch.Tensor
        分子特徴量 (必須)
    space_group : int, optional
        空間群番号 (1-230)
    density : float, optional
        目標密度 (g/cm³)
    validate_strict : bool
        厳密な検証を行うか
    
    Returns
    -------
    combined_context : torch.Tensor
        結合された条件付けコンテキスト
    """
    contexts = []
    context_names = []
    
    # 1. 分子条件付け（必須）
    if mol_features is None:
        raise ValueError('分子特徴量は必須です')
    
    mol_context = mol_cond(mol_features)
    contexts.append(mol_context)
    context_names.append(f'分子 ({mol_context.shape[-1]}次元)')
    
    # 2. 空間群条件付け（オプション）
    if space_group is not None:
        if validate_strict:
            validate_space_group(space_group)
        
        sg_tensor = torch.tensor([space_group], device=device)
        sg_context = sg_emb(sg_tensor)
        contexts.append(sg_context)
        context_names.append(f'空間群 ({sg_context.shape[-1]}次元)')
    
    # 3. 密度条件付け（オプション）
    if density is not None:
        if validate_strict:
            validate_density(density)
        
        dens_tensor = torch.tensor([density], device=device)
        dens_context = dens_cond(dens_tensor)
        contexts.append(dens_context)
        context_names.append(f'密度 ({dens_context.shape[-1]}次元)')
    
    # コンテキストの結合
    combined = torch.cat(contexts, dim=-1)
    
    return combined, context_names

# テストケース
print('条件付け組み合わせのテスト:\n')

# ダミーの分子特徴量を作成
dummy_mol_features = torch.randn(1, mol_feature_dim, device=device)

test_cases = [
    ('分子のみ', None, None),
    ('分子 + 空間群', 14, None),
    ('分子 + 密度', None, 1.2),
    ('完全条件付け', 14, 1.2),
]

for i, (name, sg, dens) in enumerate(test_cases, 1):
    try:
        combined, names = prepare_combined_conditioning(
            dummy_mol_features,
            space_group=sg,
            density=dens
        )
        
        print(f'{i}. {name}')
        print(f'   組み合わせ: {", ".join(names)}')
        print(f'   合計次元: {combined.shape[-1]}')
        print(f'   テンソル形状: {combined.shape}')
        print()
        
    except Exception as e:
        print(f'{i}. {name}')
        print(f'   ✗ エラー: {e}')
        print()

## 5. 多形生成アルゴリズム

同じ分子から異なる結晶構造（多形）を生成します。

### 5.1 多形生成戦略

In [ ]:
print('\n=== 多形生成戦略 ===\n')

# 多形生成用の空間群セット
polymorph_strategies = {
    '低対称性多形': {
        'space_groups': [1, 2, 14, 15],
        'description': '三斜晶・単斜晶系',
        'characteristics': '柔軟な分子配置、低い充填効率'
    },
    '中対称性多形': {
        'space_groups': [19, 61, 62, 33],
        'description': '斜方晶系',
        'characteristics': 'バランスの取れた対称性と充填'
    },
    '高対称性多形': {
        'space_groups': [79, 87, 225, 227],
        'description': '正方晶・立方晶系',
        'characteristics': '高度に規則的、高い充填効率'
    },
}

for strategy_name, info in polymorph_strategies.items():
    print(f'{strategy_name}:')
    print(f'  {info["description"]}')
    print(f'  空間群: {", ".join(map(str, info["space_groups"]))}')
    print(f'  特徴: {info["characteristics"]}')
    print()

def generate_polymorph_set(
    mol_features,
    space_groups,
    base_density=1.2,
    density_variation=0.1
):
    """
    多形のセットを生成するための条件を準備
    
    Parameters
    ----------
    mol_features : torch.Tensor
        分子特徴量（全多形で共通）
    space_groups : list of int
        使用する空間群のリスト
    base_density : float
        基準密度
    density_variation : float
        密度の変動幅
    
    Returns
    -------
    polymorph_conditions : list of dict
        各多形の条件付けパラメータ
    """
    polymorph_conditions = []
    
    for i, sg in enumerate(space_groups):
        # 各多形で密度を少し変化させる（厳密な制御）
        density = base_density + (i - len(space_groups)/2) * density_variation
        density = max(0.8, min(2.0, density))  # 物理的に妥当な範囲に制限
        
        try:
            combined, names = prepare_combined_conditioning(
                mol_features,
                space_group=sg,
                density=density
            )
            
            polymorph_conditions.append({
                'id': i + 1,
                'space_group': sg,
                'density': density,
                'context': combined,
                'description': f'Polymorph {i+1} (SG={sg}, ρ={density:.2f})'
            })
            
        except Exception as e:
            print(f'✗ 多形 {i+1} (SG={sg}) の準備失敗: {e}')
    
    return polymorph_conditions

# 多形生成のシミュレーション
print('\n多形生成シミュレーション:\n')

target_space_groups = [1, 14, 61, 225]  # 各結晶系から代表例を選択

polymorphs = generate_polymorph_set(
    dummy_mol_features,
    target_space_groups,
    base_density=1.2,
    density_variation=0.1
)

print(f'生成された多形条件: {len(polymorphs)}個\n')

for poly in polymorphs:
    print(f'多形 {poly["id"]}')
    print(f'  空間群: {poly["space_group"]}')
    print(f'  密度: {poly["density"]:.2f} g/cm³')
    print(f'  コンテキスト形状: {poly["context"].shape}')
    print()

print('重要: 各多形は同じ分子から生成されますが、')
print('      異なる結晶対称性と充填により、異なる構造になります。')

## 6. Wyckoff位置の取り扱い

Wyckoff位置は、空間群の対称性に基づく原子の特殊位置を表します。

### 6.1 Wyckoff位置の基本

In [ ]:
print('\n=== Wyckoff位置 ===\n')

# Wyckoff位置の例（代表的な空間群）
wyckoff_examples = {
    'P1 (1)': {
        'positions': [('a', 1, '(x, y, z)', '一般位置')],
        'note': '全て一般位置、対称性なし'
    },
    'P21/c (14)': {
        'positions': [
            ('a', 4, '(x, y, z)', '一般位置'),
            ('e', 4, '(x, 1/4, z)', '鏡映面上'),
        ],
        'note': '最頻出空間群、2つのWyckoff位置'
    },
    'Fm-3m (225)': {
        'positions': [
            ('a', 4, '(0, 0, 0)', '特殊位置（原点）'),
            ('b', 4, '(1/2, 1/2, 1/2)', '特殊位置（体心）'),
            ('c', 8, '(1/4, 1/4, 1/4)', '特殊位置'),
            ('d', 24, '(x, 0, 0)', '鏡映面上'),
            ('e', 192, '(x, y, z)', '一般位置'),
        ],
        'note': '立方晶、多数の特殊位置'
    },
}

for sg_name, info in wyckoff_examples.items():
    print(f'{sg_name}:')
    print(f'  {info["note"]}')
    print('  Wyckoff位置:')
    
    for label, multiplicity, coords, description in info['positions']:
        print(f'    {label}: 多重度={multiplicity:3d}, {coords:20s} ({description})')
    print()

print('注意事項:')
print('  • 多重度: 対称操作によって生成される等価な位置の数')
print('  • 特殊位置: 対称要素上にある位置（自由度が制限される）')
print('  • 一般位置: 対称性による制約がない位置')
print('\n重要: Wyckoff位置は厳密に守られ、近似的な配置は許されません。')

## 7. 高度なCIF操作

CIF (Crystallographic Information File) は結晶構造の標準フォーマットです。

### 7.1 CIF Writer の使用

In [ ]:
print('\n=== CIF操作 ===\n')

# CIF Writerの初期化
try:
    cif_writer = CIFWriter()
    print('✓ CIF Writer initialized')
except Exception as e:
    print(f'✗ CIF Writer initialization failed: {e}')
    cif_writer = None

if cif_writer:
    # ダミーの結晶構造データを作成
    dummy_crystal = {
        'positions': np.random.rand(10, 3) * 10.0,  # 単位セル内の座標
        'atom_types': np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0]),  # C と H
        'cell': np.array([10.0, 10.0, 10.0, 90.0, 90.0, 90.0]),  # 立方晶
        'space_group': 225,  # Fm-3m
    }
    
    print('\nCIF出力の例:\n')
    print('data_generated_crystal')
    print('_cell_length_a    10.000')
    print('_cell_length_b    10.000')
    print('_cell_length_c    10.000')
    print('_cell_angle_alpha 90.000')
    print('_cell_angle_beta  90.000')
    print('_cell_angle_gamma 90.000')
    print('_space_group_IT_number 225')
    print('_space_group_name_H-M_alt "F m -3 m"')
    print('loop_')
    print('_atom_site_label')
    print('_atom_site_type_symbol')
    print('_atom_site_fract_x')
    print('_atom_site_fract_y')
    print('_atom_site_fract_z')
    print('C1  C  0.123  0.456  0.789')
    print('H1  H  0.234  0.567  0.890')
    print('...')
    
    print('\n重要なCIFフィールド:')
    cif_fields = [
        ('_cell_length_*', '単位セルの長さ', '必須'),
        ('_cell_angle_*', '単位セルの角度', '必須'),
        ('_space_group_*', '空間群情報', '必須'),
        ('_atom_site_*', '原子座標', '必須'),
        ('_symmetry_equiv_pos_*', '対称操作', 'オプション'),
    ]
    
    for field, description, requirement in cif_fields:
        print(f'  • {field:30s} : {description:20s} ({requirement})')
    
    print('\n全てのフィールドは厳密な形式に従い、近似値は使用しません。')

## 8. 対称性解析の詳細

生成された結晶構造の対称性を解析します。

In [ ]:
print('\n=== 対称性解析 ===\n')

# SymmetryAnalyzerの使用例
try:
    symmetry_analyzer = SymmetryAnalyzer()
    print('✓ Symmetry Analyzer initialized')
    
    print('\n対称性解析で検証される項目:')
    
    analysis_items = [
        ('空間群の一致性', 'Space group consistency',
         '指定した空間群と実際の対称性が一致するか'),
        ('Wyckoff位置の妥当性', 'Wyckoff position validity',
         '原子が適切なWyckoff位置に配置されているか'),
        ('対称操作の保存', 'Symmetry operation preservation',
         '全ての対称操作が満たされているか'),
        ('単位セルの妥当性', 'Unit cell validity',
         '格子パラメータが空間群の要求を満たすか'),
        ('最小距離の確認', 'Minimum distance check',
         '原子間距離が物理的に妥当か'),
    ]
    
    for i, (name_ja, name_en, description) in enumerate(analysis_items, 1):
        print(f'\n{i}. {name_ja} ({name_en})')
        print(f'   {description}')
    
    print('\n重要:')
    print('  • 全ての検証は厳密に実行されます')
    print('  • 対称性違反は自動修正されません')
    print('  • 違反が検出された場合、構造は無効とマークされます')
    
except Exception as e:
    print(f'✗ Symmetry Analyzer initialization failed: {e}')

## 9. 統合ワークフロー

全ての機能を統合した完全なワークフローを示します。

In [ ]:
print('\n=== 統合ワークフロー ===\n')

workflow_code = '''
# complete_crystal_workflow.py
import torch
from crystal.conditioning import ExtendedCombinedConditioning
from crystal.models import CrystalDiffusion
from crystal.utils import CIFWriter
from crystal.evaluation import SymmetryAnalyzer

# 1. 条件付けの準備
mol_features = extract_molecular_features(molecule)
space_group = 14  # P21/c
target_density = 1.2  # g/cm³

# 2. 統合条件付けコンテキストの作成
conditioning = ExtendedCombinedConditioning(...)
context = conditioning(
    mol_features=mol_features,
    space_group=space_group,
    density=target_density
)

# 3. 結晶構造の生成
model = CrystalDiffusion(...)
crystal = model.sample(
    n_samples=1,
    n_atoms=100,
    context=context,
    pbc=[True, True, True]  # 周期境界条件
)

# 4. 対称性の検証（厳密）
analyzer = SymmetryAnalyzer()
is_valid, report = analyzer.validate(
    crystal,
    expected_space_group=space_group,
    strict=True  # フォールバックなし
)

if not is_valid:
    raise ValueError(f"Symmetry violation detected: {report}")

# 5. CIFファイルへの出力
writer = CIFWriter()
writer.write(
    crystal,
    filename='output.cif',
    space_group=space_group,
    validate=True  # 出力前に再検証
)

print("Crystal generated and validated successfully")
'''

print('完全なワークフロー（Pythonコード）:')
print(workflow_code)

print('\n学習用のコマンドライン例:')
train_commands = '''
# ステージ1: 分子条件付けのみで学習
python main_crystal.py \\
    --condition_on_molecule True \\
    --condition_on_space_group False \\
    --condition_on_density False \\
    --n_epochs 100 \\
    --exp_name stage1_molecule

# ステージ2: 空間群を追加
python main_crystal.py \\
    --resume outputs/stage1_molecule/model.pt \\
    --condition_on_molecule True \\
    --condition_on_space_group True \\
    --condition_on_density False \\
    --n_epochs 100 \\
    --exp_name stage2_spacegroup

# ステージ3: 密度を追加（完全条件付け）
python main_crystal.py \\
    --resume outputs/stage2_spacegroup/model.pt \\
    --condition_on_molecule True \\
    --condition_on_space_group True \\
    --condition_on_density True \\
    --n_epochs 100 \\
    --exp_name stage3_full
'''
print(train_commands)

## まとめ

### 学習した内容

✅ **複合条件付け戦略の完全実装**
   - 分子・空間群・密度の階層的組み合わせ
   - 厳密な検証とエラーハンドリング
   - 柔軟な条件付けオプション

✅ **空間群制約の厳密な適用**
   - 230個全ての空間群のサポート
   - 7つの結晶系の理解
   - 対称性の数学的定義

✅ **密度ターゲティング**
   - 物質タイプ別の典型的密度範囲
   - 厳密な密度制御
   - 物理的妥当性の検証

✅ **多形生成アルゴリズム**
   - 同一分子からの複数構造生成
   - 対称性の異なる多形
   - 系統的な多形探索

✅ **Wyckoff位置の取り扱い**
   - 特殊位置と一般位置の理解
   - 多重度の概念
   - 対称性による制約

✅ **高度なCIF操作**
   - 標準フォーマットでの出力
   - 必須フィールドの完全記述
   - 検証機能の統合

✅ **対称性解析の詳細**
   - 空間群の一致性検証
   - 対称操作の確認
   - 構造妥当性のチェック

✅ **統合ワークフロー**
   - 完全な生成パイプライン
   - 段階的な学習戦略
   - 品質保証プロセス

### 重要な原則

1. **厳密性**: 全ての対称性操作は数学的に厳密
2. **検証**: 生成後の対称性検証は必須
3. **フォールバックなし**: 対称性違反は自動修正しない
4. **透明性**: エラーは明示的に報告

### ベストプラクティス

| 項目 | 推奨事項 |
|------|----------|
| 条件付け | 常に分子条件付けを含める |
| 空間群 | 目標構造に適した対称性を選択 |
| 密度 | 物質タイプの典型値を参考に |
| 検証 | 生成後必ず対称性を確認 |
| 学習 | 段階的に条件を追加 |

### 結晶品質チェックリスト

生成された結晶構造は以下を満たす必要があります：

- [ ] 指定した空間群の対称性を持つ
- [ ] 全原子が適切なWyckoff位置に配置
- [ ] 目標密度に近い（誤差 <5%）
- [ ] 原子間距離が物理的に妥当
- [ ] 単位セルパラメータが空間群の要求を満たす
- [ ] CIFファイルが標準に準拠

### 今後の発展

このチュートリアルで学んだ技術を基に、以下の応用が可能です：

1. **医薬品結晶設計**: 溶解性・安定性の最適化
2. **材料探索**: 特定物性を持つ結晶の探索
3. **多形予測**: 実験前の多形スクリーニング
4. **構造最適化**: 目標プロパティへの最適化

### 参考資料

- **International Tables for Crystallography**: 空間群の詳細
- **CIF標準**: IUCr（国際結晶学連合）
- **対称性理論**: 群論と結晶学

---

**チュートリアルシリーズ完了**

全6つのチュートリアルを通じて、E3同変拡散モデルによる分子・結晶生成の
完全な理論と実装を学びました。

重要なのは、全てのプロセスで**厳密性**を保ち、**フォールバック処理を行わない**
ことです。これにより、生成された構造の信頼性と再現性が保証されます。